In [1]:
"""Kaggle GPU probe: epoch-2 raw checkpoint on paired raw/SRO MuSiQue dev."""

import csv
import hashlib
import json
import math
import time
from pathlib import Path

import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer


RUN_ID = "phase4b_sro_probe_20261121_01"
BASE = Path("/kaggle/input/notebooks/hungluongduc/fact-grounding-gap-deberta-full-musique")
MODEL = BASE / "fg_full_stage1/best_model_after_2_epochs"
RAW = BASE / "fg_full_data/deberta_fg_dev.csv"
SRO = Path("/kaggle/input/datasets/hungluongduc/fact-grounding-sro-phase-3-token-audit-20261121/dev.csv")
OUT = Path("/kaggle/working") / RUN_ID
EXPECTED = {
    "raw": "F7EC0B2308EBDC811BFC11828A076F498FD2CB542C503274DAD5CBA852EBD1BC",
    "sro": "2E469335EB7F58C20C9CA1225CB3E4B08338D92FC5B8B750D574C23FED85E302",
    "model": "585DAC0A9664AE69D8E218044A6C92CE83BD315820615FE0E7E27A3B01131DFC",
}


def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest().upper()


def read_csv(path):
    with path.open(encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames != ["text", "label"]:
            raise ValueError(f"invalid header: {path}")
        rows = list(reader)
    if len(rows) != 6404:
        raise ValueError(f"expected 6404 rows: {path}")
    return rows


def passages(text):
    prefix, separator = "Sub-question: ", "\n\nPassages:\n"
    if not text.startswith(prefix) or separator not in text:
        raise ValueError("invalid text contract")
    return text[len(prefix):].split(separator, 1)[1]


def metrics(labels, preds):
    tp = sum(y == 1 and p == 1 for y, p in zip(labels, preds))
    fp = sum(y == 0 and p == 1 for y, p in zip(labels, preds))
    fn = sum(y == 1 and p == 0 for y, p in zip(labels, preds))
    tn = len(labels) - tp - fp - fn
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {"rows": len(labels), "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision": precision, "recall": recall,
            "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
            "accuracy": (tp + tn) / len(labels), "flag_rate_prediction_0": (tn + fn) / len(labels)}


def main():
    checks = {"raw": sha(RAW), "sro": sha(SRO), "model": sha(MODEL / "model.safetensors")}
    if checks != EXPECTED:
        raise ValueError(f"input hash mismatch: {checks}")
    raw, sro = read_csv(RAW), read_csv(SRO)
    labels = []
    for i, (a, b) in enumerate(zip(raw, sro)):
        if a["label"] != b["label"] or a["label"] not in {"0", "1"} or passages(a["text"]) != passages(b["text"]):
            raise ValueError(f"paired row mismatch: {i}")
        labels.append(int(a["label"]))
    if labels.count(1) != 3145 or labels.count(0) != 3259:
        raise ValueError("label counts mismatch")
    if OUT.exists():
        raise FileExistsError(OUT)
    OUT.mkdir(parents=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device != "cuda":
        raise RuntimeError("phase 4B requires GPU")
    tokenizer = AutoTokenizer.from_pretrained(MODEL, local_files_only=True, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, local_files_only=True).to(device).eval()
    if model.config.label2id.get("ANSWERABLE") != 1:
        raise ValueError("wrong label mapping")
    started = time.monotonic()
    results = {}
    for name, rows in (("raw", raw), ("sro", sro)):
        probs = []
        for start in range(0, len(rows), 8):
            batch = [r["text"] for r in rows[start:start + 8]]
            x = tokenizer(batch, max_length=512, padding=True, truncation=True,
                          pad_to_multiple_of=8, return_tensors="pt").to(device)
            with torch.inference_mode():
                probs.extend(torch.softmax(model(**x).logits.float(), dim=-1)[:, 1].cpu().tolist())
            if (start // 8 + 1) % 100 == 0 or start + len(batch) == len(rows):
                print(json.dumps({"branch": name, "done": start + len(batch), "seconds": round(time.monotonic() - started, 1)}), flush=True)
        results[name] = probs
    raw_preds = [int(x >= 0.5) for x in results["raw"]]
    sro_preds = [int(x >= 0.5) for x in results["sro"]]
    raw_metric, sro_metric = metrics(labels, raw_preds), metrics(labels, sro_preds)
    if not math.isclose(raw_metric["f1"], 0.7935989549314174, abs_tol=1e-4):
        raise ValueError(f"raw checkpoint baseline mismatch: {raw_metric['f1']}")
    pred_path = OUT / "paired_predictions.csv"
    with pred_path.open("x", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["row_index", "raw_text_sha256", "sro_text_sha256", "label", "raw_prob", "sro_prob", "raw_pred", "sro_pred"])
        for i, (a, b) in enumerate(zip(raw, sro)):
            writer.writerow([i, hashlib.sha256(a["text"].encode()).hexdigest().upper(), hashlib.sha256(b["text"].encode()).hexdigest().upper(),
                             labels[i], repr(results["raw"][i]), repr(results["sro"][i]), raw_preds[i], sro_preds[i]])
    # Read back every row and metric before reporting success.
    with pred_path.open(encoding="utf-8", newline="") as f:
        paired = list(csv.DictReader(f))
    if len(paired) != 6404 or any(int(r["row_index"]) != i for i, r in enumerate(paired)):
        raise ValueError("paired prediction readback failed")
    if metrics([int(r["label"]) for r in paired], [int(r["sro_pred"]) for r in paired]) != sro_metric:
        raise ValueError("SRO metric readback failed")
    report = {"run_id": RUN_ID, "status": "completed", "checkpoint_sha256": checks["model"],
              "input_sha256": checks, "raw": raw_metric, "sro": sro_metric,
              "sro_minus_raw_f1": sro_metric["f1"] - raw_metric["f1"],
              "paired_disagreement_rows": sum(a != b for a, b in zip(raw_preds, sro_preds)),
              "predictions_sha256": sha(pred_path), "elapsed_seconds": time.monotonic() - started,
              "runtime": {"torch": torch.__version__, "transformers": transformers.__version__, "device": device},
              "interpretation": "distribution-shift probe only; not the paired retraining comparison"}
    (OUT / "metrics.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

{"branch": "raw", "done": 800, "seconds": 77.0}
{"branch": "raw", "done": 1600, "seconds": 161.2}
{"branch": "raw", "done": 2400, "seconds": 245.5}
{"branch": "raw", "done": 3200, "seconds": 331.0}
{"branch": "raw", "done": 4000, "seconds": 420.4}
{"branch": "raw", "done": 4800, "seconds": 507.5}
{"branch": "raw", "done": 5600, "seconds": 593.5}
{"branch": "raw", "done": 6400, "seconds": 680.2}
{"branch": "raw", "done": 6404, "seconds": 680.7}
{"branch": "sro", "done": 800, "seconds": 792.8}
{"branch": "sro", "done": 1600, "seconds": 903.8}
{"branch": "sro", "done": 2400, "seconds": 1014.5}
{"branch": "sro", "done": 3200, "seconds": 1126.1}
{"branch": "sro", "done": 4000, "seconds": 1240.8}
{"branch": "sro", "done": 4800, "seconds": 1354.9}
{"branch": "sro", "done": 5600, "seconds": 1465.7}
{"branch": "sro", "done": 6400, "seconds": 1578.2}
{"branch": "sro", "done": 6404, "seconds": 1578.8}
{
  "run_id": "phase4b_sro_probe_20261121_01",
  "status": "completed",
  "checkpoint_sha256": "